# Notebook 09 — Training Best Practices

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 9 of 12  
**Time estimate:** 75 minutes

> Most deep learning failures in biology are training failures, not architecture failures.
> A good debugging checklist and systematic hyperparameter protocol save weeks of
> compute time. This notebook is the practical engineering companion to the theory
> of NB01–NB08.

---
## Step 1 — Motivation

You have a DeepBind CNN (NB03). Training loss decreases but validation AUROC stays at 0.5.
Is the architecture wrong? The learning rate? The data preprocessing? The initialization?
Training best practices give you a systematic diagnostic approach — overfit one batch,
check the loss scale, inspect gradients — rather than random hyperparameter guessing.

---
## Step 2 — Intuition

**Regularization techniques:**
- **Dropout:** randomly zero out units during training with probability $p$.
  Forces the network to learn redundant representations. Equivalent to training
  an ensemble of $2^n$ sub-networks. Disable during evaluation.
- **Batch normalization:** normalize activations within a mini-batch to $\mathcal{N}(0,1)$
  at each layer, then apply learned scale $\gamma$ and shift $\beta$.
  Reduces internal covariate shift, allows higher learning rates.
- **Weight decay ($L_2$):** add $\lambda \|W\|_2^2$ to the loss.
  In Adam, use `weight_decay` parameter (AdamW is preferred — decouples decay from gradient).
- **Early stopping:** monitor validation loss; stop when it stops improving.
  Save the best checkpoint.

**Optimizers:**
- **SGD + momentum:** reliable, well-understood, may need careful LR tuning
- **Adam:** adaptive per-parameter learning rates, robust to LR choice. Default for DL in biology.
- **AdamW:** Adam + proper weight decay decoupling. Preferred for transformers.

**Learning rate scheduling:**
- **Warmup:** start with small LR, increase linearly for first $k$ steps.
  Prevents large early updates that corrupt pretrained weights.
- **Cosine decay:** decay LR following a cosine curve after warmup.
- **ReduceLROnPlateau:** cut LR by a factor when val loss stops improving.

**Gradient clipping:** clip gradient norm to max value (e.g., 1.0).
Prevents exploding gradients in RNNs and deep networks.

---
## Step 3 — Biological Background

**Debugging checklist for biological DL models:**

1. **Overfit one batch:** train on 1 batch for 100+ steps. Loss should reach ~0.
   If not: bug in forward pass, loss function, or initialization.
2. **Check loss at init:** for cross-entropy with $C$ classes, initial loss $\approx \log C$.
   For binary BCE: $\approx \log 2 \approx 0.693$. Wrong initialization = bug.
3. **Visualize the data:** are the inputs properly scaled? Is the label balance as expected?
4. **Monitor gradient norms:** `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`
   and log the pre-clip norm each epoch. If it's growing: need clipping or lower LR.
5. **Reduce, then scale:** get a working tiny model first (2 layers, 16 units),
   then scale up.

**Layer normalization vs. batch normalization in biology:**
- BatchNorm depends on batch statistics — unreliable for small batches (common in biology
  where one sample is a whole patient or a whole protein).
- LayerNorm normalizes across features within one sample — works for any batch size.
- Transformers use LayerNorm; CNNs often use BatchNorm.

**Transfer learning in biological DL:**
Pre-trained models (ESM-2, DNABERT, scGPT) have been trained on massive data.
Fine-tuning requires:
- **Lower learning rate** for pre-trained layers (1e-5 vs. 1e-3)
- **Warmup:** prevent large early updates that destroy pre-trained weights
- **Smaller batch size:** fine-tuning on small datasets needs higher gradient variance
  (less regularized than large batches)

---
## Step 4 — Mathematical Explanation

**Batch normalization:**
$$\hat{x}_i = \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$

$\mu_\mathcal{B}$, $\sigma_\mathcal{B}^2$: mean and variance over the mini-batch.
$\gamma$, $\beta$: learned scale and shift (per-feature).
At inference: use running mean and variance accumulated during training.

**Layer normalization:**
Same formula but $\mu$ and $\sigma$ are computed over the feature dimension
for each sample independently:
$$\hat{x}_{ij} = \frac{x_{ij} - \mu_i}{\sqrt{\sigma_i^2 + \epsilon}}$$

**Cosine annealing with warmup:**
$$\eta(t) = \begin{cases}
\eta_{\max} \cdot t / T_w & t < T_w \text{ (linear warmup)}\\
\eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{(t-T_w)\pi}{T - T_w}\right) & t \geq T_w
\end{cases}$$

In [ ]:
# Step 6 — Python: Systematic training best practices
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Dataset: TF binding (same as NB03) ----
NUCL = 'ACGT'
NUCL_IDX = {b: i for i, b in enumerate(NUCL)}
MOTIF = 'CCGCGNGGNGG'; MOTIF_L = 11; SEQ_L = 50
rng = np.random.default_rng(42)

def generate_seqs(n=1000):
    seqs, labels = [], []
    for i in range(n):
        seq = list(''.join(rng.choice(list(NUCL), SEQ_L)))
        if i < n//2:
            pos = rng.integers(0, SEQ_L-MOTIF_L)
            for j, b in enumerate(MOTIF):
                if b != 'N': seq[pos+j] = b if rng.random()>0.1 else rng.choice(list(NUCL))
            labels.append(1)
        else:
            labels.append(0)
        seqs.append(''.join(seq))
    return seqs, np.array(labels)

def one_hot_batch(sequences):
    L = len(sequences[0])
    X = np.zeros((len(sequences), 4, L), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j, b in enumerate(seq):
            if b in NUCL_IDX: X[i, NUCL_IDX[b], j] = 1.0
    return X

seqs, y_seq = generate_seqs(1000)
X_oh = one_hot_batch(seqs)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X); self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

full_ds = SeqDS(X_oh, y_seq)
train_ds, val_ds = torch.utils.data.random_split(full_ds, [800, 200])
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=200)

# ---- CNN architectures with different regularization ----
def make_cnn(dropout=0.0, use_batchnorm=False):
    layers = [nn.Conv1d(4, 16, 11, padding=0)]
    if use_batchnorm: layers.append(nn.BatchNorm1d(16))
    layers.extend([nn.ReLU(), nn.AdaptiveMaxPool1d(1), nn.Flatten()])
    if dropout > 0: layers.append(nn.Dropout(dropout))
    layers.extend([nn.Linear(16, 32), nn.ReLU()])
    if dropout > 0: layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(32, 1))
    layers.append(nn.Sigmoid())
    return nn.Sequential(*layers)

configs = [
    ('No regularization', dict(dropout=0.0, use_batchnorm=False)),
    ('Dropout (p=0.3)', dict(dropout=0.3, use_batchnorm=False)),
    ('BatchNorm', dict(dropout=0.0, use_batchnorm=True)),
    ('BatchNorm + Dropout', dict(dropout=0.3, use_batchnorm=True)),
]

# ---- Train all configs ----
from sklearn.metrics import roc_auc_score

results = {}
for name, cfg in configs:
    model = make_cnn(**cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCELoss()
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
    
    train_losses, val_aurocs, grad_norms = [], [], []
    for epoch in range(50):
        model.train()
        bl, gn_bl = [], []
        for X_b, y_b in train_ld:
            optimizer.zero_grad()
            pred = model(X_b.to(device))
            loss = criterion(pred, y_b.to(device))
            loss.backward()
            # Track gradient norm before clipping
            total_norm = sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None)
            total_norm = total_norm.item()**0.5
            gn_bl.append(total_norm)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            bl.append(loss.item())
        scheduler.step()
        train_losses.append(np.mean(bl))
        grad_norms.append(np.mean(gn_bl))
        
        model.eval()
        y_probs, y_true = [], []
        with torch.no_grad():
            for X_b, y_b in val_ld:
                y_probs.extend(model(X_b.to(device)).cpu().numpy().ravel())
                y_true.extend(y_b.numpy().ravel())
        auroc = roc_auc_score(y_true, y_probs)
        val_aurocs.append(auroc)
    
    results[name] = {'losses': train_losses, 'aurocs': val_aurocs, 'grads': grad_norms}
    print(f'{name}: final val AUROC={max(val_aurocs):.4f}')

# ---- Demonstrate: overfit one batch check ----
model_debug = make_cnn(dropout=0.0, use_batchnorm=False).to(device)
opt_debug = optim.Adam(model_debug.parameters(), lr=1e-2)
X_one_batch, y_one_batch = next(iter(train_ld))
X_one_batch, y_one_batch = X_one_batch.to(device), y_one_batch.to(device)
batch_losses = []
for _ in range(100):
    opt_debug.zero_grad()
    loss = nn.BCELoss()(model_debug(X_one_batch), y_one_batch)
    loss.backward(); opt_debug.step()
    batch_losses.append(loss.item())
print(f'\nOverfit check: initial loss={batch_losses[0]:.4f} (expected ~0.693 for binary)')
print(f'After 100 steps on same batch: {batch_losses[-1]:.4f} (should be near 0)')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = ['grey', 'steelblue', 'green', 'tomato']

# Panel A: Val AUROC by regularization config
ax = axes[0]
for (name, _), color in zip(configs, colors):
    ax.plot(results[name]['aurocs'], color=color, lw=1.5, label=f'{name.split(" (")[0]}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val AUROC')
ax.set_title('A. Regularization comparison\n(val AUROC over training)')
ax.legend(fontsize=7)

# Panel B: Train loss comparison
ax = axes[1]
for (name, _), color in zip(configs, colors):
    ax.plot(results[name]['losses'], color=color, lw=1.5, label=name.split(' (')[0])
ax.set_xlabel('Epoch'); ax.set_ylabel('Train loss')
ax.set_title('B. Training loss\n(BatchNorm enables faster convergence)')
ax.legend(fontsize=7)

# Panel C: Gradient norm over training
ax = axes[2]
for (name, _), color in zip(configs, colors):
    ax.plot(results[name]['grads'], color=color, lw=1.2, alpha=0.8, label=name.split(' (')[0])
ax.axhline(1.0, color='k', ls='--', lw=0.8, label='Clip threshold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Gradient L2 norm')
ax.set_title('C. Gradient norms\n(clip threshold = 1.0)')
ax.legend(fontsize=7)

# Panel D: Cosine LR schedule
ax = axes[3]
T_max = 50
lr_schedule = []
for t in range(T_max):
    if t < 5:  # warmup
        lr = 1e-3 * t / 5
    else:
        lr = 1e-5 + 0.5*(1e-3 - 1e-5)*(1 + math.cos((t-5)*math.pi/(T_max-5)))
    lr_schedule.append(lr)
ax.plot(range(T_max), lr_schedule, 'tomato', lw=2)
ax.axvspan(0, 5, alpha=0.15, color='steelblue', label='Warmup (5 epochs)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning rate')
ax.set_title('D. Cosine LR schedule\n(warmup + cosine decay)')
ax.legend(fontsize=8)

plt.suptitle('Module 14 NB09: Training Best Practices', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('training_best_practices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Run the overfit-one-batch check on the NB03 DeepBind CNN. If you see the loss
   not reaching near-zero in 100 steps, identify and fix the bug.
2. Implement a learning rate warmup manually (without a scheduler class):
   multiply the LR by `min(1, step/warmup_steps)` at each step. Show the
   effect on early training stability.
3. Compare Adam vs. AdamW (adds proper weight decay decoupling). On which datasets
   does the difference matter?
4. Show what happens to batch normalization at inference if you forget to call
   `model.eval()`. Why does this cause a performance drop?

---
## Step 10 — Quiz

1. What does batch normalization compute? What are $\gamma$ and $\beta$?
2. Why does dropout not help in all cases? When should you use weight decay instead?
3. What is gradient clipping? Why is it important for RNNs but less so for CNNs?
4. What is the warmup period in a learning rate schedule and why is it needed
   for fine-tuning pre-trained models?
5. Describe the overfit-one-batch debugging technique. What failure modes does it catch?

---
## Step 12 — Reflection

> *[In one paragraph: explain why batch normalization can fail for biological datasets
> where batch size is forced to be small (e.g., a single patient profile per sample),
> and describe what alternative you would use and why.]*

---
*Next: `10_model_interpretability.ipynb`*